In [11]:
import pandas as pd
import plotly.express as px
import requests
from fredapi import Fred
import plotly.graph_objects as go
import dash
from dash import html, dcc, Input, Output, callback
import json
import Essential_methods_variables as emv
import plotly.express as px
import importlib
import numpy as np

importlib.reload(emv)



<module 'Essential_methods_variables' from 'C:\\Users\\dsvin\\Desktop\\economic-industry-dashboard\\notebooks\\Essential_methods_variables.py'>

In [2]:
#df_orders, df_inventories, and df_interest are the dataframes to extract code data from 
emv.df_orders.columns

Index(['Level', 'Measure', 'FRED API Code'], dtype='object')

In [23]:
df_total_OI = pd.concat([emv.df_orders[emv.df_orders['Level']==0],emv.df_inventories[emv.df_inventories['Level']==0]])
df_one_OI = pd.concat([emv.df_orders[emv.df_orders['Level']==1],emv.df_inventories[emv.df_inventories['Level']==1]])

df_two_OI = pd.concat([emv.df_orders[emv.df_orders['Level']==2],emv.df_inventories[emv.df_inventories['Level']==2]])


def update_measure(row):
    code = str(row["FRED API Code"])
    measure = row["Measure"]
    if code.endswith("I"):
        return "Inv: " + measure
    else:
        return measure

df_total_OI["Measure"] = df_total_OI.apply(update_measure,axis =1)
df_one_OI["Measure"] =  df_one_OI.apply(update_measure,axis =1)
df_two_OI["Measure"] =  df_two_OI.apply(update_measure,axis =1)

def data_clean_matrix(bracket,matrix=True):
    check = []
    x = pd.DataFrame()
    for i,j in zip(bracket["FRED API Code"],bracket["Measure"]):
        y = emv.data_cleaning(i)
        y.rename(columns={'Value':j},inplace=True)
        check.append(y.shape[0])
        if x.empty:
            x = y 
        else:
            x = x.merge(y, on = 'Date')
    print(f"{max(check)-min(check)} monthly datapoint(s) are being lost in the process")
    if matrix:
        corr_df = x[bracket["Measure"]]
        corr_matrix = corr_df.corr(method = 'pearson')
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        corr_mask = corr_matrix.mask(mask)
        
        fig  = px.imshow(
            corr_matrix,
            text_auto=True,
            color_continuous_scale='RdBu_r',
            title='Correlation Heatmap:',
            zmin = -1, zmax=1
        )
        fig.update_layout(height=900,width=900)
        fig.update_traces(textfont_size=5)
        return fig
        
    return x

level_0 = data_clean_matrix(df_total_OI)
level_1 = data_clean_matrix(df_one_OI)
level_2 = data_clean_matrix(df_two_OI)



1 monthly datapoint(s) are being lost in the process
1 monthly datapoint(s) are being lost in the process
2 monthly datapoint(s) are being lost in the process


In [14]:
#Insight: The demand for computer and electronic product is not correlated with any other demand or inventory amount,
#It is almost as its own thing, which is really interesting. Except for inventory of comp and electric products. (For level 2)

#Insight: For level 1, it seems like Durable, Nondurable and its correponding inventories move almost in unison with each other,
#with a high correlation

#Level 0 is the same as level 1. 


In [25]:
app  = dash.Dash(__name__)

server = app.server

app.layout = html.Div(children=[
    html.H1(children="Movement Correlations of Inventories and New Orders"),
    dcc.Dropdown(id='pearson',options = [
        {"label":"Aggregate Inventories and Orders","value":"agg"},
        {"label":"Durable and Non-Durable Goods","value":"dnd"},
        {"label":"Durable Goods subcomponents","value":"durable"}    
    ],placeholder="select which correlations you would like to see"),
    html.Div(id="graph"),html.Div()
])

@app.callback(
    Output('graph','children'),
    Input('pearson','value')
)
def update_graph(select):
    if select == "agg":
        return dcc.Graph(figure=level_0)
    elif select == "dnd":
        return dcc.Graph(figure=level_1)
    elif select == "durable":
        return dcc.Graph(figure=level_2)
    else:
        return html.Div()
    

if __name__ == '__main__':
    app.run(debug=True,port = 8051)